<a href="https://colab.research.google.com/github/gussoares/consulta_pje/blob/main/api_pje_trts_3.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

### API de consulta pública ao PJE com usuário e senha

In [ ]:
import requests
import json

In [ ]:
username = ''  # insira o usuário (CPF)
password = "" # insira a senha do usuario
payload = {'login':username,'senha':password } # transforma em dicionário
data=json.dumps(payload) # transforma em json

In [ ]:
url9 = 'https://pje.trt2.jus.br/pje-consulta-api/api/auth' # endpoint que autentica

In [ ]:
headers = {'Accept-Encoding': 'gzip, deflate, br', 'Content-type': 'application/json','x-grau-instancia' : str(1)}
r = requests.post(url9, data, headers=headers)

In [ ]:
# pega o token de acesso
def pega_access_token(usuario, senha, tribunal = '2', instancia = '1'):
    payload = {'login' : usuario, 'senha' : senha}
    headers = {'Accept-Encoding': 'gzip, deflate, br', 'Content-type': 'application/json','x-grau-instancia' : str(instancia)}
    url9 = 'https://pje.trt' + str(tribunal) + '.jus.br/pje-consulta-api/api/auth' # auth
    r = requests.post(url9, data, headers=headers)
    access_token = r.json()['access_token']
    return access_token

In [ ]:
# TESTE
def pega_access_token2(usuario, senha, tribunal = '2', instancia = '1'):
    payload = {'username' : usuario, 'password' : senha, 'novaSenha1': '', 'novaSenha2' : ""}
    headers = {'Accept': 'text/html,application/xhtml+xml,application/xml;q=0.9,image/avif,image/webp,*/*;q=0.8',
               'Accept-Encoding': 'gzip, deflate, br',
               'Accept-Language' : "pt-BR,pt;q=0.8,en-US;q=0.5,en;q=0.3",
               'Content-type': 'application/x-www-form-urlencoded',
               'Cache-Control' : 'no-cache',
               'Connection' : 'keep-alive',
               'Content-Length' : '66',
               'Host' : 'pje.trt2.jus.br',
               'Origin' : 'https://pje.trt2.jus.br',
               'Pragma' : 'no-cache',
               'Referer' : 'https://pje.trt2.jus.br/primeirograu/login.seam',
               'Sec-Fetch-Dest' : 'document',
               'Sec-Fetch-Mode' : 'navigate',
               'Sec-Fetch-Site' :'same-origin',
               'Sec-Fetch-User' : "?1",
               'TE' : 'trailers',
               'Upgrade-Insecure-Requests' : '1',
               'User-Agent' : 'Mozilla/5.0 (Windows NT 10.0; Win64; x64; rv:108.0) Gecko/20100101 Firefox/108.0'
                }
    url9 = 'https://pje.trt2.jus.br/primeirograu/logar.seam'
    url8 = 'https://pje.trt2.jus.br/primeirograu/login.seam?cid=195645'
    #'https://pje.trt' + str(tribunal) + '.jus.br/pje-comum-api/api/auth' # auth
    #r1 = requests.post(url8, data, headers=headers)
    r1 = requests.post(url8, data, headers=headers)
    #access_token = r.json()['access_token']
    #'Xsrf-Token' : r1.cookies['Xsrf-Token'],
    cookies = r1.cookies #{'JSESSIONID' : r1.cookies['JSESSIONID'], 'br.com.infox.ibpm.skin' : "skin/azul"}
    print(cookies)
    r = requests.post(url8, cookies, data, headers=headers)
    return r1

In [ ]:
autenticacao = pega_access_token2(username, password)

<RequestsCookieJar[<Cookie JSESSIONID=4af043250571c861~OkIBcPhLEdLEDt7OArSCXBLy for pje.trt2.jus.br/primeirograu>, <Cookie br.com.infox.ibpm.skin=skin/azul for pje.trt2.jus.br/primeirograu>]>


In [ ]:
#autenticacao.cookies['JSESSIONID']


autenticacao.headers


{'x-request-id': 'd55bcaeb-d094-4418-8eed-6265c56009f0', 'content-type': 'application/json', 'content-length': '1620', 'date': 'Tue, 24 Jan 2023 20:06:17 GMT', 'strict-transport-security': 'max-age=15768000; includeSubDomains; preload', 'cache-control': 'public, no-transform', 'content-security-policy': "frame-ancestors 'self' jte.csjt.jus.br jte.trt5.jus.br *.jte.trt5.jus.br", 'x-content-type-options': 'nosniff', 'referrer-policy': 'no-referrer-when-downgrade', 'feature-policy': "camera 'none'; microphone 'none'", 'access-control-expose-headers': 'Date'}

In [ ]:
autenticacao

<Response [200]>

In [ ]:
# funcao generica que a partir do número cnj permite pergar o número id

def pega_id(numero_processo_cnj, tribunal = '2', instancia = '1'):
    headers = {'x-grau-instancia' : str(instancia)}
    url = "https://pje.trt" + str(tribunal) + ".jus.br/pje-consulta-api/api/processos/dadosbasicos/" + numero_processo_cnj
    r = requests.get(url, headers=headers)
    return r.json()[0]['id']

In [ ]:
def consulta_autenticada(numero_processo_id, access_token, tribunal = '2', instancia = '1'):
    #authorization = {'Authorization' : autenticacao.json()['token_type'] + " " + autenticacao.json()['access_token']}
    headers = {'Authorization': 'Bearer {}'.format(access_token), 'Accept': 'application/json, text/plain, */*', 'Accept-Encoding': 'gzip, deflate, br', 'Content-type': 'application/json','x-grau-instancia' : str(instancia)}
    url2 = "https://pje.trt" + str(tribunal) + ".jus.br/pje-consulta-api/api/processos/" + str(numero_processo_id)
    r = requests.get(url2, headers=headers)
    return r

In [ ]:
def baixa_integra_pje_autenticado(numero_processo_id, access_token, tribunal = '2', instancia = '1'):
    headers = {'Authorization': 'Bearer {}'.format(access_token),  'Accept': 'application/json, text/plain, */*', 'Accept-Encoding': 'gzip, deflate, br', 'Content-type': 'application/json','x-grau-instancia' : str(instancia)}
    url3a = "https://pje.trt" + str(tribunal) + ".jus.br/pje-consulta-api/api/processos/" + str(numero_processo_id) +'/integra'
    r2 = requests.get(url3a, headers=headers)
    nome_do_arquivo = str(numero_processo_id) + "_" + 'integra'
    if r2.content.startswith(b'%PDF'):
        with open(f'{nome_do_arquivo}.pdf', 'wb') as f:
            f.write(r2.content)
    else:
        baixa_integra_pje_autenticado(numero_processo_id, access_token, tribunal = '2', instancia = '1')





In [ ]:
exemplo = '1000238-08.2022.5.02.0371' # insira o número cnj de um processo

In [ ]:
# para pegar o número id
id = pega_id(exemplo)

In [ ]:
id

4425861

In [ ]:
# para pegar o token de acesso
autenticacao = pega_access_token(username, password)

In [ ]:
autenticacao

'eyJhbGciOiJIUzI1NiJ9.eyJqdGkiOiIyZDAwZDY5Zi1kMTc5LTQ2NDUtYWY1NS1iZjUwOTlkZjAxYjciLCJpc3MiOiJwamUtc2VndXJhbmNhIiwiaWF0IjoxNjc0NTkwOTcwLCJzdWIiOiIxMmVlYTMwNi0xYjhjLTQyZTYtOTFlMy1jMGVlZjQxYWUwZTguY3B1IiwiZXhwIjoxNjc0NTk0NTcwLCJzaWdsYVNpc3RlbWEiOiJDUFUiLCJpbnN0YW5jaWEiOjEsInRpcG8iOiJVU1VBUklPIiwiaWQiOjE1MDA3NDQsImxvZ2luIjoiOTM3Mjk4ODAwNjMiLCJub21lIjoiR1VTVEFWTyBTQ0hJTEQgU09BUkVTIiwicGVyZmlsIjoyNTEzNTUsInBhcGVsIjp7ImlkIjoxNDY5LCJub21lIjoiTWFnaXN0cmFkbyIsImlkZW50aWZpY2Fkb3IiOiJNQUdJU1RSQURPIn0sInBhcGVsS3oiOnsiaWQiOjE5LCJub21lIjoiTWFnaXN0cmFkbyIsImlkZW50aWZpY2Fkb3IiOiJNQUdJU1RSQURPIn0sImxvY2FsaXphY2FvIjp7ImlkIjoyODM0LCJkZXNjcmljYW8iOiIxwqogVmFyYSBkbyBUcmFiYWxobyBkZSBNb2dpIGRhcyBDcnV6ZXMifSwibWV0b2RvQXV0ZW50aWNhY2FvIjoiTE9HSU5fU0VOSEEiLCJvcmdhb0p1bGdhZG9yIjp7ImlkIjo2MywiZGVzY3JpY2FvIjoiMcKqIFZhcmEgZG8gVHJhYmFsaG8gZGUgTW9naSBkYXMgQ3J1emVzIn0sIm9yaWdlbSI6IlRSVDIifQ.zXCaHSd6JsBRdlIMIdi619eLnIu1yvIGC0zVD_T-htw'

In [ ]:
# exemplo de consulta por número id com autenticacao
consulta_autenticada_exemplo = consulta_autenticada(id, autenticacao)

In [ ]:
consulta_autenticada_exemplo.json()

{'id': 4425861,
 'numero': '1000238-08.2022.5.02.0371',
 'classe': 'ATOrd',
 'orgaoJulgador': '1ª Vara do Trabalho de Mogi das Cruzes',
 'segredoJustica': False,
 'justicaGratuita': True,
 'distribuidoEm': '2022-02-25T21:47:44.495',
 'autuadoEm': '2022-02-25T21:47:44.366',
 'valorDaCausa': 53807.0,
 'poloAtivo': [{'id': 23393984,
   'idPessoa': 5655474,
   'nome': 'EDILBERTO GALVAO DA HORA',
   'login': '19144130406',
   'tipo': 'RECLAMANTE',
   'documento': '191.441.304-06',
   'tipoDocumento': 'CPF',
   'endereco': {'logradouro': 'RUA BENTO RAMOS DE QUEIROZ ',
    'numero': '251',
    'bairro': 'JARDIM SANTA CAROLINA',
    'municipio': 'MOGI DAS CRUZES',
    'estado': 'SP',
    'cep': '08770-130'},
   'polo': 'ATIVO',
   'situacao': 'ATIVO',
   'representantes': [{'id': 24738800,
     'idPessoa': 18220,
     'nome': 'SILVIO LUIS BIROLLI',
     'login': '01122969805',
     'tipo': 'ADVOGADO',
     'documento': '011.229.698-05',
     'tipoDocumento': 'CPF',
     'endereco': {'logradour

In [ ]:
def baixa_integra_pje_autenticado(numero_processo_id, access_token, tribunal = '2', instancia = '1'):
    headers = {'Authorization': 'Bearer {}'.format(access_token),  'Accept': 'application/json, text/plain, */*', 'Accept-Encoding': 'gzip, deflate, br', 'Content-type': 'application/json','x-grau-instancia' : str(instancia)}
    url3a = "https://pje.trt" + str(tribunal) + ".jus.br/pje-consulta-api/api/processos/" + str(numero_processo_id) +'/integra'
    r2 = requests.get(url3a, headers=headers)
    nome_do_arquivo = str(numero_processo_id) + "_" + 'integra'
    if r2.content.startswith(b'%PDF'):
        with open(f'{nome_do_arquivo}.pdf', 'wb') as f:
            f.write(r2.content)
    else:
        baixa_integra_pje_autenticado(numero_processo_id, access_token, tribunal = '2', instancia = '1')

In [ ]:
# exemplo para baixar a integra de um processo
baixa_integra_pje_autenticado(pega_id(exemplo), autenticacao)

KeyboardInterrupt: 

In [ ]:
def orgaos_julgadores(access_token, tribunal = '2', instancia = '1'):
    headers = {'Authorization': 'Bearer {}'.format(access_token),  'Accept': 'application/json, text/plain, */*', 'Accept-Encoding': 'gzip, deflate, br', 'Content-type': 'application/json','x-grau-instancia' : str(instancia)}
    url = "https://pje.trt" + str(tribunal) + ".jus.br/pje-consulta-api/api/orgaosjulgadores?somenteOJCs=true"
    r2 = requests.get(url, headers=headers)
    return r2

In [ ]:
def classes_processuais
/pje-consulta-api/api/classes

In [ ]:
/pje-consulta-api/api/assuntos

In [ ]:
oj = orgaos_julgadores(autenticacao)

In [ ]:
for i in oj.json():
    if i['id'] == 63:
        print(i)

In [ ]:
#data no formato YYYY-MM-DD
def consulta_pauta(data, access_token, tribunal = '2', instancia = '1', pagina = '1', tamanhoPagina = "20", ordenacaoColuna = 'undefined', ordenacaoCrescente = 'undefined', idOj = '63'):
    headers = {'Authorization': 'Bearer {}'.format(access_token),  'Accept': 'application/json, text/plain, */*', 'Accept-Encoding': 'gzip, deflate, br', 'Content-type': 'application/json','x-grau-instancia' : str(instancia)}
    url = "https://pje.trt" + str(tribunal) + ".jus.br/pje-consulta-api/api/audiencias?pagina=" + pagina + '&tamanhoPagina=' + tamanhoPagina + '&ordenacaoColuna=' + ordenacaoColuna + '&ordenacaoCrescentes=' + ordenacaoCrescente + "&idOj=" + idOj + '&data=' + data
    r2 = requests.get(url, headers=headers)
    return r2
#https://pje.trt2.jus.br/pje-consulta-api/api/audiencias?pagina=1&tamanhoPagina=20&ordenacaoColuna=undefined&ordenacaoCrescente=undefined&idOj=63&data=2023-01-23

In [ ]:
data_exemplo = '2023-01-23'

In [ ]:
pauta = consulta_pauta(data_exemplo, autenticacao)

In [ ]:
pauta.json()


{'pagina': 1,
 'tamanhoPagina': 20,
 'qtdPaginas': 1,
 'totalRegistros': 10,
 'resultado': [{'indice': 1,
   'data': '2023-01-23T08:10:00',
   'idProcesso': 4759641,
   'classeProcesso': 'ATOrd',
   'numeroProcesso': '1001121-52.2022.5.02.0371',
   'sala': 'sala padrão',
   'tipo': 'Una por videoconferência',
   'status': 'DESIGNADA',
   'poloAtivo': [{'nome': 'CLAUDIO JOSE DE SOUZA',
     'utilizaLoginSenha': False}],
   'resumoPoloAtivo': 'CLAUDIO JOSE DE SOUZA',
   'poloPassivo': [{'nome': 'ANDERSON PATRICK DA SILVA - ME',
     'utilizaLoginSenha': False}],
   'resumoPoloPassivo': 'ANDERSON PATRICK DA SILVA - ME',
   'juizoDigital': True},
  {'indice': 2,
   'data': '2023-01-23T08:30:00',
   'idProcesso': 4747015,
   'classeProcesso': 'ATOrd',
   'numeroProcesso': '1001087-77.2022.5.02.0371',
   'sala': 'sala padrão',
   'tipo': 'Una por videoconferência',
   'status': 'DESIGNADA',
   'poloAtivo': [{'nome': 'NATHALIA GUEDES DA SILVA',
     'utilizaLoginSenha': False}],
   'resumoPol

In [ ]:
for i in pauta.json()['resultado']:
    print(i['numeroProcesso'])

1001121-52.2022.5.02.0371
1001087-77.2022.5.02.0371
1000508-32.2022.5.02.0371
1001127-59.2022.5.02.0371
1001094-69.2022.5.02.0371
1001274-85.2022.5.02.0371
1001276-55.2022.5.02.0371
1001277-40.2022.5.02.0371
1001280-92.2022.5.02.0371
1001287-84.2022.5.02.0371


In [ ]:
def download_pauta(data, access_token, tribunal = '2', instancia = '1', pagina = '1', tamanhoPagina = "20", ordenacaoColuna = 'undefined', ordenacaoCrescente = 'undefined', idOj = '63'):
    pauta = consulta_pauta(data=data, access_token =access_token, tribunal = tribunal, instancia = instancia, pagina = pagina, tamanhoPagina =  tamanhoPagina, ordenacaoColuna = ordenacaoColuna, ordenacaoCrescente = ordenacaoCrescente, idOj = idOj):
    for numero_processo_cnj in pauta.json()['resultado']:
        numero_processo_id = pega_id(numero_processo_cnj, tribunal = tribunal, instancia = instancia):
        baixa_integra_pje_autenticado(numero_processo_id, access_token, tribunal = tribunal, instancia = instancia):

    headers = {'Authorization': 'Bearer {}'.format(access_token),  'Accept': 'application/json, text/plain, */*', 'Accept-Encoding': 'gzip, deflate, br', 'Content-type': 'application/json','x-grau-instancia' : str(instancia)}


    url = "https://pje.trt" + str(tribunal) + ".jus.br/pje-consulta-api/api/audiencias?pagina=" + pagina + '&tamanhoPagina=' + tamanhoPagina + '&ordenacaoColuna=' + ordenacaoColuna + '&ordenacaoCrescentes=' + ordenacaoCrescente + "&idOj=" + idOj + '&data=' + data
    r2 = requests.get(url, headers=headers)
    return r2

In [ ]:

#data no formato YYYY-MM-DD
# máximo permitido 185 dias
# classe_processo 387 sum 326 ord
def consulta_terceiros(cnpj_parte, classe_processo, data_inicio, data_fim, access_token, tribunal = '2', instancia = '1', pagina = '1', tamanhoPagina = "20", ordenacaoColuna = 'undefined', ordenacaoCrescente = 'undefined', idOj = '63'):
    headers = {'Authorization': 'Bearer {}'.format(access_token),  'Accept': 'application/json, text/plain, */*', 'Accept-Encoding': 'gzip, deflate, br', 'Content-type': 'application/json','x-grau-instancia' : str(instancia)}
    uri = "https://pje.trt" + str(tribunal) + ".jus.br/pje-consulta-api/api/processos/consultaterceiros?pagina=" + pagina + '&tamanhoPagina=' + tamanhoPagina + '&ordenacaoColuna=' + ordenacaoColuna + '&ordenacaoCrescente=' + ordenacaoCrescente + '&cnpjParte=' + cnpj_parte + '&idClasseProcesso=' + classe_processo + '&dtDistribuicaoInicio=' + data_inicio + 'T00:00:00&' + 'dtDistribuicaoFim=' + data_fim + 'T23:59:59'
    r2 = requests.get(uri, headers=headers)
    return r2
#&idClasseProcesso=387 (ATSum) 326 (ordinario)

In [ ]:
cnpj_pesquisa = '24232886000167' # necessariamente sem pontos, travessão e barra
data_inicio_pesquisa = "2021-01-01"
data_fim_pesquisa = "2021-07-01"
classe_processo = '326' #'387'
c =  consulta_terceiros(cnpj_pesquisa, classe_processo, data_inicio_pesquisa, data_fim_pesquisa, autenticacao, tribunal = '2', instancia = '1', pagina = '1', tamanhoPagina = "20", ordenacaoColuna = 'undefined', ordenacaoCrescente = 'undefined')
c.json()

{'pagina': 1,
 'tamanhoPagina': 20,
 'qtdPaginas': 1,
 'totalRegistros': 16,
 'resultado': [{'id': 3916129,
   'numeroIdentificacaoJustica': 502,
   'numero': '1000363-08.2021.5.02.0016',
   'classe': 'ATOrd',
   'orgaoJulgador': '16ª Vara do Trabalho de São Paulo',
   'segredoJustica': False,
   'justicaGratuita': True,
   'distribuidoEm': '2021-03-29T16:09:59.562',
   'autuadoEm': '2021-03-29T16:09:59.429',
   'valorDaCausa': 114787.34,
   'poloAtivo': [{'id': 20624305,
     'idPessoa': 735280,
     'nome': 'FABIOLA PARISI CURCI FUIM',
     'login': '26982993809',
     'tipo': 'RECLAMANTE',
     'documento': '269.829.938-09',
     'tipoDocumento': 'CPF',
     'endereco': {'logradouro': 'PRACA PADRE MARIO FONTANA ',
      'numero': '94',
      'complemento': 'APARTAMENTO 203',
      'bairro': 'PARQUE DA MOOCA',
      'municipio': 'SAO PAULO',
      'estado': 'SP',
      'cep': '03127-030'},
     'polo': 'ATIVO',
     'situacao': 'ATIVO',
     'representantes': [{'id': 20624131,
      

In [ ]:
id = c.json()['resultado'][1]['id']
id

KeyError: 'resultado'

In [ ]:
consulta_cnpj = consulta_autenticada(id, autenticacao)

In [ ]:
consulta_cnpj.json()

{'id': 3961846,
 'numero': '1000512-06.2021.5.02.0371',
 'classe': 'ATOrd',
 'orgaoJulgador': '2ª Vara do Trabalho de Mogi das Cruzes',
 'segredoJustica': False,
 'justicaGratuita': True,
 'distribuidoEm': '2021-05-10T15:44:59.895',
 'autuadoEm': '2021-05-10T15:44:59.636',
 'valorDaCausa': 39574.39,
 'poloAtivo': [{'id': 21013889,
   'idPessoa': 3288671,
   'nome': 'GILDETE DE SOUZA CORDEIRO',
   'login': '26060237819',
   'tipo': 'RECLAMANTE',
   'documento': '260.602.378-19',
   'tipoDocumento': 'CPF',
   'endereco': {'logradouro': 'RUA EXPEDICIONARIO FRANCISCO ANTONIO DE OLIVEIRA ',
    'numero': '145',
    'complemento': 'bloco 4, apto. 23',
    'bairro': 'JARDIM ESPERANCA',
    'municipio': 'MOGI DAS CRUZES',
    'estado': 'SP',
    'cep': '08743-580'},
   'polo': 'ATIVO',
   'situacao': 'ATIVO',
   'representantes': [{'id': 21013858,
     'idPessoa': 69073,
     'nome': 'VICENTE APARECIDO LOPES DA SILVA',
     'login': '14900839876',
     'tipo': 'ADVOGADO',
     'documento': '14

In [ ]:
    processos/consultaterceiros?pagina=1&tamanhoPagina=20&ordenacaoColuna=undefined&ordenacaoCrescente=undefined&cnpjParte=03022122000177&dtDistribuicaoInicio=2022-11-01T00:00:00&dtDistribuicaoFim=2022-12-16T23:59:59

    https://pje.trt2.jus.br/pje-consulta-api/api/processos/consultaterceiros?pagina=1&tamanhoPagina=20&ordenacaoColuna=undefined&ordenacaoCrescente=undefined&cnpjParte=03022122000177&dtDistribuicaoInicio=2022-11-01T00:00:00&dtDistribuicaoFim=2022-12-16T23:59:59
    https://pje.trt2.jus.br/pje-consulta-api/api/processos/consultaterceiros?pagina=1&tamanhoPagina=20&ordenacaoColuna=undefined&ordenacaoCrescente=undefined&cpfParte=&oabAdvProcesso=SP-305874&dtDistribuicaoInicio=2021-12-25T00:00:00&dtDistribuicaoFim=2022-12-16T23:59:59
    r2 = requests.get(url, headers=headers)
    return r2

SyntaxError: invalid syntax (Temp/ipykernel_35920/3880766080.py, line 1)